In [21]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler

In [22]:
TRAIN_data = r"C:\Users\hp\Desktop\Data_analysis-and-ML\train.csv"
TEST_data = r"C:\Users\hp\Desktop\Data_analysis-and-ML\test.csv"
SUBMISSION_data = r"C:\Users\hp\Desktop\Data_analysis-and-ML\sample_submission.csv"

train = pd.read_csv(TRAIN_data)
test = pd.read_csv(TEST_data)

print(train.shape, test.shape)
train.head()

(1241, 81) (219, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,...,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,985,90,RL,75.0,10125,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,Mitchel,Norm,Norm,Duplex,1.5Fin,5,5,1977,1977,Gable,CompShg,Plywood,Plywood,NaN,0.0,TA,TA,CBlock,NaN,NaN,NaN,NaN,0,NaN,0,0,0,GasA,...,Y,SBrkr,1302,432,0,1742,0,0,2,0,4,2,Gd,8,Typ,0,NaN,Attchd,1977.0,Unf,2,539,TA,TA,Y,0,0,0,0,0,0,NaN,NaN,NaN,0,8,2009,COD,Normal,125969
1,778,20,RL,100.0,13350,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,Sawyer,Norm,Norm,1Fam,1Story,5,5,1974,1974,Hip,CompShg,HdBoard,Plywood,NaN,0.0,TA,TA,CBlock,TA,TA,No,ALQ,762,Unf,0,102,864,GasA,...,Y,SBrkr,894,0,0,885,1,0,1,0,3,1,TA,5,Typ,1,Fa,Attchd,1974.0,Unf,2,440,TA,TA,Y,241,0,0,0,0,0,NaN,MnPrv,NaN,0,6,2006,WD,Normal,142555
2,708,120,RL,48.0,6240,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,NridgHt,Norm,Norm,TwnhsE,1Story,8,5,2006,2006,Hip,CompShg,MetalSd,MetalSd,BrkFace,176.0,Gd,TA,PConc,Gd,TA,No,GLQ,863,Unf,0,461,1324,GasA,...,Y,SBrkr,1324,0,0,1323,1,0,2,0,2,1,Gd,6,Typ,1,Gd,Attchd,2006.0,Fin,2,550,TA,TA,Y,192,38,0,0,0,0,NaN,NaN,NaN,0,12,2009,WD,Normal,254214
3,599,20,RL,80.0,12984,Pave,NaN,Reg,Bnk,AllPub,Inside,Gtl,Crawfor,Norm,Norm,1Fam,1Story,5,6,1977,1977,Gable,CompShg,Plywood,Plywood,BrkFace,459.0,TA,TA,CBlock,Gd,TA,Mn,ALQ,1283,LwQ,147,0,1430,GasA,...,Y,SBrkr,1647,0,0,1614,1,0,2,0,3,1,Gd,7,Typ,1,TA,Attchd,1977.0,Fin,2,621,TA,TA,Y,0,0,0,0,0,0,NaN,NaN,NaN,0,3,2006,WD,Normal,217309
4,875,50,RM,52.0,5720,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,OldTown,Artery,Norm,1Fam,1.5Fin,5,6,1941,1950,Gable,CompShg,Wd Sdng,Wd Sdng,NaN,0.0,TA,TA,CBlock,TA,TA,No,Unf,0,Unf,0,676,676,GasA,...,Y,SBrkr,676,455,0,1149,0,0,1,1,3,1,TA,5,Typ,0,NaN,Detchd,1941.0,Unf,1,200,TA,TA,Y,26,0,0,0,0,0,NaN,NaN,NaN,0,8,2009,WD,Abnorml,66406


In [23]:
train = train.drop_duplicates()
test_ID = test["Id"]

def features(df):
    df = df.copy()

    if "Id" in df.columns:
        df = df.drop(columns=["Id"])

    na_as_none_cols = [
        "Alley", "BsmtQual", "BsmtCond", "BsmtExposure",
        "BsmtFinType1", "BsmtFinType2", "FireplaceQu",
        "GarageType", "GarageFinish", "GarageQual", "GarageCond",
        "PoolQC", "Fence", "MiscFeature"
    ]

    for col in na_as_none_cols:
        if col in df.columns:
            df[col] = df[col].fillna("None")

    zero_fill_cols = [
        "GarageYrBlt", "GarageArea", "GarageCars",
        "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
        "BsmtFullBath", "BsmtHalfBath", "MasVnrArea"
    ]

    for col in zero_fill_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)
    
    if {"1stFlrSF", "2ndFlrSF", "TotalBsmtSF"}.issubset(df.columns):
        df["TotalSF"] = df["1stFlrSF"] + df["2ndFlrSF"] + df["TotalBsmtSF"]

    if {"FullBath", "HalfBath", "BsmtFullBath", "BsmtHalfBath"}.issubset(df.columns):
        df["TotalBath"] = (
            df["FullBath"]
            + 0.5 * df["HalfBath"]
            + df["BsmtFullBath"]
            + 0.5 * df["BsmtHalfBath"]
        )

    if {"YrSold", "YearBuilt"}.issubset(df.columns):
        df["HouseAge"] = df["YrSold"] - df["YearBuilt"]

    if {"YrSold", "YearRemodAdd"}.issubset(df.columns):
        df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]

    if {"GrLivArea", "LotArea"}.issubset(df.columns):
        df["LivLotRatio"] = df["GrLivArea"] / (df["LotArea"] + 1)

    if {"OverallQual", "OverallCond"}.issubset(df.columns):
        df["TotalHomeQuality"] = df["OverallQual"] * df["OverallCond"]

    if {"OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch"}.issubset(df.columns):
        df["TotalPorchSF"] = (
            df["OpenPorchSF"]
            + df["EnclosedPorch"]
            + df["3SsnPorch"]
            + df["ScreenPorch"]
        )

    if "GarageArea" in df.columns:
        df["HasGarage"] = (df["GarageArea"] > 0).astype(int)

    if "TotalBsmtSF" in df.columns:
        df["HasBsmt"] = (df["TotalBsmtSF"] > 0).astype(int)

    if "Fireplaces" in df.columns:
        df["HasFireplace"] = (df["Fireplaces"] > 0).astype(int)

    if "PoolArea" in df.columns:
        df["HasPool"] = (df["PoolArea"] > 0).astype(int)

    if "2ndFlrSF" in df.columns:
        df["Has2ndFloor"] = (df["2ndFlrSF"] > 0).astype(int)

    if {"WoodDeckSF", "OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch"}.issubset(df.columns):
        df["TotalOutdoorSF"] = (  
            df["WoodDeckSF"]
            + df["OpenPorchSF"]
            + df["EnclosedPorch"]
            + df["3SsnPorch"]
            + df["ScreenPorch"]
        )

    if {"TotRmsAbvGrd", "KitchenAbvGr"}.issubset(df.columns):
        df["TotalRooms"] = df["TotRmsAbvGrd"] + df["KitchenAbvGr"]  

    if {"GarageArea", "GarageCars"}.issubset(df.columns):
        df["GarageScore"] = df["GarageArea"] * df["GarageCars"]  

    if {"OverallQual", "GrLivArea"}.issubset(df.columns):
        df["QualArea"] = df["OverallQual"] * df["GrLivArea"]  

    if {"YearRemodAdd", "YearBuilt"}.issubset(df.columns):
        df["IsRemodeled"] = (df["YearRemodAdd"] > df["YearBuilt"]).astype(int)  

    if {"YrSold", "YearBuilt"}.issubset(df.columns):
        df["IsNewHouse"] = (df["YrSold"] == df["YearBuilt"]).astype(int)  

    if {"TotalBath", "TotRmsAbvGrd"}.issubset(df.columns):
        df["BathPerRoom"] = df["TotalBath"] / (df["TotRmsAbvGrd"] + 1)  

    if {"OverallQual", "TotalSF"}.issubset(df.columns):
        df["Qual_x_TotalSF"] = df["OverallQual"] * df["TotalSF"]  

    if {"HouseAge", "OverallQual"}.issubset(df.columns):
        df["Age_x_Qual"] = df["HouseAge"] * df["OverallQual"]  

    skewed_cols = ["LotArea", "GrLivArea", "TotalBsmtSF", "1stFlrSF", "MasVnrArea"]  
    for col in skewed_cols:  
        if col in df.columns:  
            df[col] = np.log1p(df[col])  

    if "OverallQual" in df.columns:
        df["OverallQual2"] = df["OverallQual"] ** 2  

    if "TotalSF" in df.columns:
        df["TotalSF2"] = df["TotalSF"] ** 2  

    return df


def group_rare_categories(train_df, test_df, min_count=10):  
    train_df = train_df.copy()  
    test_df = test_df.copy()  

    cat_cols = train_df.select_dtypes(include=["object"]).columns.tolist()  

    for col in cat_cols:  
        freq = train_df[col].value_counts()  
        rare_values = freq[freq < min_count].index  

        train_df[col] = train_df[col].replace(rare_values, "Rare")  
        test_df[col] = test_df[col].replace(rare_values, "Rare")  

    return train_df, test_df  

def log_transform_skewed(train_df, test_df, threshold=0.75):
    train_df = train_df.copy()
    test_df = test_df.copy()

    num_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()

    skewness = train_df[num_cols].skew().sort_values(ascending=False)
    skewed_cols = skewness[skewness.abs() > threshold].index.tolist()

    for col in skewed_cols:
        if col in train_df.columns and col in test_df.columns:
            if train_df[col].min() >= 0 and test_df[col].min() >= 0:
                train_df[col] = np.log1p(train_df[col])
                test_df[col] = np.log1p(test_df[col])

    return train_df, test_df, skewed_cols

train_fe = features(train)
test_fe = features(test)

X = train_fe.drop(columns=["SalePrice"])
y = train_fe["SalePrice"]
X_test = test_fe.copy()

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

In [24]:
def make_preprocessor(X, scaler_type="robust"):
    cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
    num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

    if scaler_type == "robust":
        scaler = RobustScaler()
    elif scaler_type == "standard":
        scaler = StandardScaler()
    else:
        scaler = "passthrough"

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", scaler)
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, num_cols),
        ("cat", categorical_pipeline, cat_cols)
    ])

    return preprocessor

In [25]:
def cv_mae_original_scale(model, X, y, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = []

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X), start=1):
        X_train, X_valid = X.iloc[tr_idx], X.iloc[va_idx]
        y_train, y_valid = y.iloc[tr_idx], y.iloc[va_idx]

        m = clone(model)
        m.fit(X_train, np.log1p(y_train))

        preds = np.expm1(m.predict(X_valid))
        preds = np.clip(preds, 10000, None)

        mae = mean_absolute_error(y_valid, preds)
        scores.append(mae)

        print(f"Fold {fold}: {mae:,.2f}")

    print("Mean CV MAE:", f"{np.mean(scores):,.2f}")
    print("Std CV MAE: ", f"{np.std(scores):,.2f}")
    return np.mean(scores), np.std(scores)

In [26]:
preprocessor = make_preprocessor(X, scaler_type="robust")

alphas = [0.0001, 0.0003, 0.0005, 0.001, 0.003, 0.01]
l1_ratios = [0.05, 0.1, 0.15, 0.2, 0.4, 0.6]

results = []

for alpha in alphas:
    for l1_ratio in l1_ratios:
        model = Pipeline([
            ("preprocessor", preprocessor),
            ("regressor", ElasticNet(
                alpha=alpha,
                l1_ratio=l1_ratio,
                max_iter=30000,
                random_state=42
            ))
        ])

        mean_mae, std_mae = cv_mae_original_scale(model, X, y)

        results.append({
            "alpha": alpha,
            "l1_ratio": l1_ratio,
            "mean_mae": mean_mae,
            "std_mae": std_mae
        })

results_df = pd.DataFrame(results).sort_values("mean_mae")
print(results_df.head(10))

Fold 1: 15,362.32
Fold 2: 14,537.42
Fold 3: 15,246.12
Fold 4: 14,521.70
Fold 5: 15,361.46
Mean CV MAE: 15,005.80
Std CV MAE:  391.18
Fold 1: 15,327.58
Fold 2: 14,495.21
Fold 3: 15,237.23
Fold 4: 14,411.76
Fold 5: 15,289.09
Mean CV MAE: 14,952.17
Std CV MAE:  409.04
Fold 1: 15,313.16
Fold 2: 14,479.62
Fold 3: 15,215.14
Fold 4: 14,302.19
Fold 5: 15,246.05
Mean CV MAE: 14,911.23
Std CV MAE:  429.70
Fold 1: 15,298.76
Fold 2: 14,472.35
Fold 3: 15,204.39
Fold 4: 14,214.81
Fold 5: 15,207.57
Mean CV MAE: 14,879.57
Std CV MAE:  446.44
Fold 1: 15,296.60
Fold 2: 14,384.81
Fold 3: 15,105.13
Fold 4: 13,847.30
Fold 5: 15,088.60
Mean CV MAE: 14,744.49
Std CV MAE:  545.46
Fold 1: 15,192.23
Fold 2: 14,353.61
Fold 3: 14,885.37
Fold 4: 13,490.77
Fold 5: 14,989.32
Mean CV MAE: 14,582.26
Std CV MAE:  612.09
Fold 1: 15,369.95
Fold 2: 14,349.63
Fold 3: 15,076.89
Fold 4: 13,605.69
Fold 5: 15,203.30
Mean CV MAE: 14,721.09
Std CV MAE:  657.61
Fold 1: 15,331.33
Fold 2: 14,291.50
Fold 3: 15,022.07
Fold 4: 13,443.

In [27]:
best_alpha = results_df.iloc[0]["alpha"]
best_l1_ratio = results_df.iloc[0]["l1_ratio"]

preprocessor = make_preprocessor(X, scaler_type="robust")

elastic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", ElasticNet(
        alpha=best_alpha,
        l1_ratio=best_l1_ratio,
        max_iter=30000,
        random_state=42
    ))
])

ridge_alphas = [1.0, 3.0, 5.0, 10.0, 20.0, 30.0]
ridge_results = []

for alpha in ridge_alphas:
    preprocessor = make_preprocessor(X, scaler_type="robust")

    ridge_model = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", Ridge(alpha=alpha))
    ])

    mean_mae, std_mae = cv_mae_original_scale(ridge_model, X, y)

    ridge_results.append({
        "alpha": alpha,
        "mean_mae": mean_mae,
        "std_mae": std_mae
    })
ridge_results_df = pd.DataFrame(ridge_results).sort_values("mean_mae")
print(ridge_results_df)

elastic_model.fit(X, np.log1p(y))

elastic_preds = np.expm1(elastic_model.predict(X_test))
elastic_preds = np.clip(elastic_preds, 10000, None)

best_ridge_alpha = ridge_results_df.iloc[0]["alpha"]

preprocessor = make_preprocessor(X, scaler_type="robust")

ridge_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", Ridge(alpha=best_ridge_alpha))
])

ridge_model.fit(X, np.log1p(y))

ridge_preds = np.expm1(ridge_model.predict(X_test))
ridge_preds = np.clip(ridge_preds, 10000, None)

mean_mae, std_mae = cv_mae_original_scale(elastic_model, X, y)
print("Best ElasticNet CV MAE:", mean_mae)

final_preds = 0.6 * elastic_preds + 0.4 * ridge_preds

Fold 1: 15,420.48
Fold 2: 14,358.43
Fold 3: 14,914.48
Fold 4: 13,148.12
Fold 5: 15,113.34
Mean CV MAE: 14,590.97
Std CV MAE:  800.19
Fold 1: 15,347.25
Fold 2: 14,360.34
Fold 3: 14,751.06
Fold 4: 12,835.78
Fold 5: 14,869.31
Mean CV MAE: 14,432.75
Std CV MAE:  858.34
Fold 1: 15,308.04
Fold 2: 14,381.71
Fold 3: 14,726.26
Fold 4: 12,819.33
Fold 5: 14,796.70
Mean CV MAE: 14,406.41
Std CV MAE:  846.99
Fold 1: 15,347.89
Fold 2: 14,464.62
Fold 3: 14,806.28
Fold 4: 12,800.27
Fold 5: 14,800.70
Mean CV MAE: 14,443.95
Std CV MAE:  869.23
Fold 1: 15,530.79
Fold 2: 14,575.27
Fold 3: 15,114.82
Fold 4: 12,841.73
Fold 5: 15,031.15
Mean CV MAE: 14,618.75
Std CV MAE:  938.90
Fold 1: 15,753.29
Fold 2: 14,761.17
Fold 3: 15,518.91
Fold 4: 12,931.09
Fold 5: 15,242.13
Mean CV MAE: 14,841.32
Std CV MAE:  1,010.63
   alpha      mean_mae      std_mae
2    5.0  14406.408539   846.994736
1    3.0  14432.747430   858.342453
3   10.0  14443.950841   869.225864
0    1.0  14590.970218   800.186604
4   20.0  14618.7512

In [28]:
submission = pd.DataFrame({
    "Id": test_ID,
    "SalePrice": final_preds
})

submission.to_csv("submission_ML4.csv", index=False)
print("submission.csv created")

submission.csv created
